# Streamlit 질문·승인 화면

화면은 `uv run --extra notebooks streamlit run src/courseware/app.py`로 엽니다. 이 노트북은 서버를 계속 띄우지 않고 AppTest로 실제 화면 로직을 검증합니다. 기본 앱은 외부 API를 호출하지 않습니다.

구현 정본은 [app.py](app.py)입니다. 아래에서 현재 파일을 읽고 import하므로 Python 수정이 자동으로 반영됩니다. 수업 본문은 [통합 교안](../../lessons/)을 사용하세요.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/courseware").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"


In [ ]:
from IPython.display import display, Code
display(Code((ROOT / "src/courseware/app.py").read_text(encoding="utf-8"), language="python"))

## 작은 실행과 검토

입력·출력·실패 조건을 예상하고 실행하세요. 원문 코드의 구현과 실제 결과를 대조합니다.

In [ ]:
from streamlit.testing.v1 import AppTest
app = AppTest.from_file(str(ROOT / "src/courseware/app.py"), default_timeout=30).run()
app.chat_input[0].set_value("RAG 검색").run()
assert not app.exception
assert "[D1]" in app.session_state["messages"][-1]["content"]
app.radio[0].set_value("승인 후 보고서").run()
app.button[0].click().run()
assert app.session_state["pending"]
next(b for b in app.button if b.label == "승인하고 작성").click().run()
assert not app.exception
assert app.session_state["result"]["writer_executed"]
print("질문·조사·승인·작성 화면 PASS")